# Step 1: download sequence reads from ENA

This notebook downloads raw sequence reads (FASTQ files) for one record in the
[European Nucleotide Archive (ENA)](https://www.ebi.ac.uk/ena/browser/home).

What it does, in order:

1. You paste in an ENA accession.
2. It asks ENA which FASTQ files exist for that accession.
3. It shows you the files and their sizes. Nothing is downloaded yet.
4. It downloads the files into the `data/` folder.
5. It checks each downloaded file is complete.
6. It prints the first read so you can see what is inside.

Run the cells one at a time, from top to bottom, and read the output of each before moving on.

## 1. Imports

Four tools are needed:

- `requests` sends a request to a web address and brings back the answer.
- `Path` builds file and folder paths.
- `hashlib` calculates the MD5 of a file. An MD5 is a short fingerprint of a file's contents.
- `gzip` reads compressed `.gz` files.

`requests` is the only one that had to be installed. The other three come with Python.

In [ ]:
import gzip
import hashlib
from pathlib import Path

import requests

## 2. Your input: the ENA accession

Paste the accession between the quotes.

- A **run** accession (starts with `ERR`, `SRR` or `DRR`) is one sequencing run. This is the simplest case.
- A **sample** accession (`SAMEA...`) or a **study** accession (`PRJEB...`) also works. ENA then returns every run that belongs to it, which can be a lot of files.

In [ ]:
ACCESSION = ""

print("Accession:", ACCESSION)

## 3. Ask ENA which files exist

This sends one request to the **ENA Portal API**. The part of the API used here is called the *file report*. It answers the question: which files does ENA hold for this accession?

The request has four settings:

- `accession`: the record you pasted in above.
- `result`: `read_run` means we want sequencing runs.
- `fields`: the columns we want back.
  - `run_accession`: the run's ID.
  - `fastq_ftp`: where the FASTQ files are.
  - `fastq_md5`: the MD5 fingerprint of each file.
  - `fastq_bytes`: the size of each file in bytes.
- `format`: `json` is a format Python can read directly.

`timeout=60` means: give up if ENA has not answered after 60 seconds.

**Look at the output.** `HTTP status: 200` means ENA answered properly. Any other number means something went wrong, and you should stop here. A status of `500` means the ENA service itself has a problem.

In [ ]:
ENA_FILE_REPORT_URL = "https://www.ebi.ac.uk/ena/portal/api/filereport"

settings = {
    "accession": ACCESSION,
    "result": "read_run",
    "fields": "run_accession,fastq_ftp,fastq_md5,fastq_bytes",
    "format": "json",
}

response = requests.get(ENA_FILE_REPORT_URL, params=settings, timeout=60)

print("Address asked:", response.url)
print("HTTP status:", response.status_code)

## 4. Read ENA's answer

`response.raise_for_status()` stops the notebook with an error if the status above was not a success.

`response.json()` turns ENA's answer into a Python list called `runs`. Each item in the list is one run.

**Look at the output.** You should see one line per run. If a run has two files (paired-end sequencing), the two are joined with a `;`.

In [ ]:
response.raise_for_status()

runs = response.json()

print("Number of runs found:", len(runs))

for run in runs:
    print(run)

## 5. List the files and their sizes

**Nothing is downloaded in this cell.** It only builds a tidy list of the files so you can check the size first.

ENA gives the files of one run as a single piece of text joined with `;`. This cell splits that text so there is one entry per file.

ENA gives the file location without `https://` at the front, so the cell adds it.

Some runs have no FASTQ files at ENA. The cell says so and moves to the next run.

Lines worth explaining:

- `continue` means: skip the rest of this turn of the loop and go to the next run.
- `.split(";")` cuts a piece of text at every `;` and gives back a list.
- `zip(locations, md5s, sizes)` walks through the three lists side by side, so the first location goes with the first MD5 and the first size.
- `location.split("/")[-1]` cuts the location at every `/` and keeps the last part, which is the file name.
- `int(size)` turns the size from text into a number so it can be added up.

**Look at the output.** Check the total size at the bottom against your free disk space before you run the next cell.

In [ ]:
files = []

for run in runs:

    if run["fastq_ftp"] == "":
        print(run["run_accession"], "has no FASTQ files at ENA")
        continue

    locations = run["fastq_ftp"].split(";")
    md5s = run["fastq_md5"].split(";")
    sizes = run["fastq_bytes"].split(";")

    for location, md5, size in zip(locations, md5s, sizes):
        file = {
            "run": run["run_accession"],
            "url": "https://" + location,
            "name": location.split("/")[-1],
            "md5": md5,
            "bytes": int(size),
        }
        files.append(file)

total_bytes = 0

for file in files:
    print(file["run"], file["name"], round(file["bytes"] / 1000000, 1), "MB")
    print("   ", file["url"])
    total_bytes = total_bytes + file["bytes"]

print()
print("Number of files:", len(files))
print("Total download size:", round(total_bytes / 1000000000, 2), "GB")

## 6. Download the files

The files are saved in `data/<your accession>/`. The folder is created if it does not exist.

Each file is fetched in pieces of 1 MB and written to disk piece by piece. This avoids holding a whole large file in memory.

If a file with the same name is already in the folder, it is skipped.

Lines worth explaining:

- `stream=True` tells `requests` to hand the file over piece by piece instead of all at once.
- `download.raise_for_status()` stops with an error if the file server did not answer with a success.
- `open(local_path, "wb")` opens a new file for writing. `w` is write, `b` is bytes (raw data, not text).

**Careful:** if a download is interrupted, a partial file is left behind, and running this cell again will skip it. The next cell catches that. If the next cell reports `MISMATCH` for a file, delete that file and run this cell again.

In [ ]:
DATA_FOLDER = Path("data") / ACCESSION
DATA_FOLDER.mkdir(parents=True, exist_ok=True)

ONE_MB = 1024 * 1024

for file in files:

    local_path = DATA_FOLDER / file["name"]

    if local_path.exists():
        print("Already here, skipping:", local_path)
        continue

    print("Downloading:", file["url"])

    download = requests.get(file["url"], stream=True, timeout=60)
    download.raise_for_status()

    with open(local_path, "wb") as output_file:
        for piece in download.iter_content(chunk_size=ONE_MB):
            output_file.write(piece)

    print("Saved:", local_path)

print("Finished.")

## 7. Check the downloaded files are complete

ENA told us the MD5 fingerprint of each file in step 3. This cell calculates the MD5 of each file now on your disk and compares the two.

- `OK` means the file on your disk is identical to the file at ENA.
- `MISMATCH` means the file is incomplete or damaged. Delete it and run the download cell again.

Lines worth explaining:

- `open(local_path, "rb")` opens the file for reading. `r` is read, `b` is bytes.
- The file is read in pieces of 1 MB. When a piece comes back empty (`b""`), the end of the file has been reached and `break` leaves the loop.
- `md5_calculator.update(piece)` feeds one piece into the fingerprint calculation.
- `md5_calculator.hexdigest()` gives the finished fingerprint as text.

In [ ]:
for file in files:

    local_path = DATA_FOLDER / file["name"]

    md5_calculator = hashlib.md5()

    with open(local_path, "rb") as input_file:
        while True:
            piece = input_file.read(ONE_MB)
            if piece == b"":
                break
            md5_calculator.update(piece)

    local_md5 = md5_calculator.hexdigest()

    if local_md5 == file["md5"]:
        print("OK      ", local_path)
    else:
        print("MISMATCH", local_path)
        print("    ENA says:  ", file["md5"])
        print("    Your file: ", local_md5)

## 8. Look inside the first file

A FASTQ file stores each read as four lines:

1. The read's name, starting with `@`.
2. The sequence (A, C, G, T, and N for an unknown base).
3. A `+` line.
4. The quality of each base, one character per base.

This cell opens the first downloaded file and prints its first four lines, which is one read.

Lines worth explaining:

- `files[0]` is the first file in the list. Python counts from 0.
- `gzip.open(first_file, "rt")` opens the compressed file for reading as text. `r` is read, `t` is text.
- `line.strip()` removes the invisible end-of-line character so the output has no blank lines.

In [ ]:
first_file = DATA_FOLDER / files[0]["name"]

print("File:", first_file)
print()

with gzip.open(first_file, "rt") as input_file:
    for line_number in range(4):
        line = input_file.readline()
        print(line.strip())